# 01 — Load Silver Data

**J&J HRD 2030 Predictive Hiring Demo**

This notebook transforms the raw Bronze tables into clean, typed Silver tables following the Medallion Architecture pattern.

### What this notebook does
1. Reads each Bronze table (all-STRING columns)
2. Applies type casts, string trimming, null filtering, and deduplication
3. Adds audit columns (e.g. `ingested_at`)
4. Writes the result as Delta Silver tables in the same UC schema
5. Validates row counts and displays sample rows for each Silver table

### Prerequisites
- `00_load_bronze.ipynb` must have run successfully
- Bronze tables exist in the target catalog/schema

**Next notebook:** `01_1_vector_search.ipynb`

## Setup — Configuration Parameters

Read catalog and schema widget values, set the default Spark context, and import transformation libraries.

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "bx4",      "UC Catalog")
dbutils.widgets.text("schema",  "hrd_2030", "UC Schema")

In [ ]:
# ---------------------------------------------------------------------------
# Retrieve widget values and set the default catalog/schema for the session
# ---------------------------------------------------------------------------
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

# Set the default catalog and database for the session so we can use short table names
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Active context: {catalog}.{schema}")
print(f"Timestamp     : {datetime.now().isoformat()}")

## Transform Bronze → Silver: Candidates

Cast numeric columns (scores, years, budget) from STRING to INT/DOUBLE, trim whitespace, drop rows with null `candidate_id`, and deduplicate. New candidates (C011–C020) retain NULLs for post-interview score columns.

In [ ]:
# ---------------------------------------------------------------------------
# Silver: candidates
# Includes embedded feature scores (education_score … culture_fit) and hired label.
# New candidates (C011-C020) have nulls for skills_match_score, interview_score,
# culture_fit, and hired — preserved intentionally for ML inference use case.
# ---------------------------------------------------------------------------
print("Building candidates...")

bronze_candidates = spark.table("bronze_candidates")

def trim_string_cols(df):
    from pyspark.sql.types import StringType
    str_cols = [c.name for c in df.schema.fields if isinstance(c.dataType, StringType)]
    for col in str_cols:
        df = df.withColumn(col, F.trim(F.col(col)))
    return df

SCORE_COLS = [
    "education_score", "experience_score", "leadership_score",
    "certification_score", "skills_match_score", "industry_relevance_score",
    "interview_score", "culture_fit",
]

silver_candidates = (
    bronze_candidates
    .transform(trim_string_cols)
    .filter(F.col("candidate_id").isNotNull() & (F.col("candidate_id") != ""))
    .dropDuplicates(["candidate_id"])
    # Profile numeric columns
    .withColumn("years_total_experience",  F.col("years_total_experience").cast(IntegerType()))
    .withColumn("years_hr_experience",     F.col("years_hr_experience").cast(IntegerType()))
    .withColumn("years_leadership",        F.col("years_leadership").cast(IntegerType()))
    .withColumn("direct_reports_managed",  F.col("direct_reports_managed").cast(IntegerType()))
    .withColumn("budget_managed_millions", F.col("budget_managed_millions").cast(DoubleType()))
    # Feature score columns — cast to INT, preserve nulls for new candidates
    .withColumn("hired", F.col("hired").cast(IntegerType()))
)

for col in SCORE_COLS:
    silver_candidates = silver_candidates.withColumn(col, F.col(col).cast(IntegerType()))

silver_candidates = silver_candidates.withColumn("ingested_at", F.current_timestamp())

(
    silver_candidates
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{catalog}`.`{schema}`.candidates")
)

total  = spark.table("candidates").count()
scored = spark.sql(f"SELECT COUNT(*) AS n FROM `{catalog}`.`{schema}`.candidates WHERE interview_score IS NOT NULL").collect()[0]["n"]
print(f"✓ candidates written — {total:,} total rows ({scored} with complete features, {total-scored} new/unscored)")

## Transform Bronze → Silver: Job Requirements

Apply the same cleanup pattern to job requirements — cast salary and team size columns, trim strings, and deduplicate on `job_id`.

In [ ]:
# ---------------------------------------------------------------------------
# Silver: job_requirements
# ---------------------------------------------------------------------------
print("Building job_requirements...")

bronze_job_req = spark.table("bronze_job_requirements")

silver_job_req = (
    bronze_job_req
    .transform(trim_string_cols)
    .filter(F.col("job_id").isNotNull() & (F.col("job_id") != ""))
    .dropDuplicates(["job_id"])
    # Cast integer fields
    .withColumn("min_years_experience",  F.col("min_years_experience").cast(IntegerType()))
    .withColumn("min_hr_years",          F.col("min_hr_years").cast(IntegerType()))
    .withColumn("min_leadership_years",  F.col("min_leadership_years").cast(IntegerType()))
    .withColumn("team_size",             F.col("team_size").cast(IntegerType()))
    # Cast double (currency) fields
    .withColumn("salary_min", F.col("salary_min").cast(DoubleType()))
    .withColumn("salary_max", F.col("salary_max").cast(DoubleType()))
    # Audit column
    .withColumn("ingested_at", F.current_timestamp())
)

(
    silver_job_req
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{catalog}`.`{schema}`.job_requirements")
)

print(f"✓ job_requirements written — {spark.table('job_requirements').count():,} rows")

## Validate Silver Tables

Print row counts and display sample rows for each Silver table to confirm the transformations applied correctly.

In [ ]:
# ---------------------------------------------------------------------------
# Row counts and schema verification for Silver tables
# ---------------------------------------------------------------------------
silver_tables = {
    "candidates":       ["candidate_id", "first_name", "last_name", "job_id",
                         "education_score", "interview_score", "culture_fit", "hired"],
    "job_requirements": ["job_id", "title", "location", "min_years_experience",
                         "salary_min", "salary_max", "team_size"],
}

print("=" * 60)
print("Silver Table Validation")
print("=" * 60)

for tbl, sample_cols in silver_tables.items():
    df    = spark.table(tbl)
    count = df.count()
    print(f"\n{'─' * 60}")
    print(f"Table : {tbl}")
    print(f"Rows  : {count:,}")
    print(f"\nSample rows:")
    df.select(*[c for c in sample_cols if c in df.columns]).show(5, truncate=60)

print("\n✓ Silver validation complete.")

## Next Steps

Silver tables are now available with proper types and clean data:

| Table | Key Changes from Bronze |
|---|---|
| `candidates` | Numeric columns cast to INT/DOUBLE; null `candidate_id` rows removed; deduped |
| `job_requirements` | Salary/team_size cast to DOUBLE/INT; null `job_id` rows removed; deduped |
| `training_data` | All score columns cast to INT; `hired` flag cast to INT (0/1); deduped |

All tables include an `ingested_at` audit timestamp.

**Proceed to** `01_1_vector_search.ipynb` to extract resume text from PDFs and build a Vector Search index for semantic candidate retrieval.